In [ ]:
import sys
import pandas as pd
from pathlib import Path

# Add src to path to allow imports from src
ROOT_DIR = Path.cwd().parent.parent
if str(ROOT_DIR) not in sys.path:
    sys.path.append(str(ROOT_DIR))

In [ ]:
from src.data.file_loader import load_csv_data
from src.models.const import REPORTS_DIR, SUMMARY_CSV_NAME, RegressionMetrics
from src.utils.const import ALPHA_SIGNIFICANCE_LEVEL
from src.models.evaluation_utils import load_ablation_study_metrics, get_metric_matrix, aggregate_metrics_for_main_effects
from src.models.evaluation import check_global_normality, perform_friedman_analysis, perform_conover_posthoc

STUDY_RESULTS_FILEPATH = REPORTS_DIR / SUMMARY_CSV_NAME

# Column names from the experiment summary CSV
ALGORITHM_COL = "Algorithm"
VARIANT_COL = "Variant"
DURATION_COL = "Duration (s)"
FEATURES_COUNT_COL = "Features_Count"

# Metric columns to be analyzed
METRIC_COLS = [
    "mean_mae",
    "mean_mse",
    "mean_rmse",
    "std_mae",
    "std_mse",
    "std_rmse"
]

COLS_TO_ANALYZE = [DURATION_COL] + METRIC_COLS

# Aggregation functions to apply
AGGREGATION_METHODS = ["mean", "min", "max"]

### 1. General descriptive statistics

The following functions provide a descriptive statistics summary of the experiment metrics, grouped by the main effects of **Algorithm** and **Variant**.

*Note: The results that are being grouped are already aggregated, providing a single metric per model rather than individual outer cross-validation folds.*

In [ ]:
def _calculate_group_statistics(
    df: pd.DataFrame, 
    group_by_column: str, 
    target_columns: list[str],
    aggregation_methods: list[str] = AGGREGATION_METHODS
) -> pd.DataFrame:
    """
    Calculate mean, min, and max for specified columns grouped by a given key.

    Args:
        df: The input DataFrame containing ablation study results.
        group_by_column: The column name to group by (e.g., Algorithm or Variant).
        target_columns: List of columns to aggregate.
        aggregation_methods: List of aggregation methods to apply (default: ["mean", "min", "max"]).

    Returns:
        pd.DataFrame: Aggregated statistics with multi-index columns.
    """
    return df.groupby(group_by_column)[target_columns].agg(aggregation_methods)

In [ ]:
def get_algorithm_performance_stats(
    df: pd.DataFrame,
    cols_to_analyze: list[str] = COLS_TO_ANALYZE
) -> pd.DataFrame:
    """
    Compute performance statistics grouped by Algorithm.

    Calculate mean, min, and max for duration and all metric columns 
    for each algorithm in the dataset.

    Args:
        df: The input DataFrame containing ablation study results.
        cols_to_analyze: List of columns to analyze (default: COLS_TO_ANALYZE).

    Returns:
        pd.DataFrame: Aggregated statistics for each algorithm.
    """
    return _calculate_group_statistics(df, ALGORITHM_COL, cols_to_analyze)


def get_variant_performance_stats(
    df: pd.DataFrame,
    cols_to_analyze: list[str] = COLS_TO_ANALYZE
) -> pd.DataFrame:
    """
    Compute performance statistics grouped by Variant.

    Calculates mean, min, and max for duration and all metric columns 
    for each model variant in the dataset.

    Args:
        df: The input DataFrame containing ablation study results.
        cols_to_analyze: List of columns to analyze (default: COLS_TO_ANALYZE).

    Returns:
        pd.DataFrame: Aggregated statistics for each variant.
    """
    return _calculate_group_statistics(df, VARIANT_COL, cols_to_analyze)

In [ ]:
# Load the ablation study results
df_results = load_csv_data(STUDY_RESULTS_FILEPATH)

# Get statistics grouped by Algorithm
algo_stats = get_algorithm_performance_stats(df_results)
print("Algorithm metrics summary:")
display(algo_stats)

# Get statistics grouped by Variant
variant_stats = get_variant_performance_stats(df_results)
print("Variant metrics summary:")
display(variant_stats)

### 2. Results preparation for statistical analysis

This section handles the preparation of experiment results for subsequent statistical testing. Unlike the previous section, these results are non-aggregated, preserving the individual metric values across all outer cross-validation folds for each model.

While matrices for MAE, MSE, and RMSE are extracted here, only MAE is selected and used for the remaining analysis.

In [ ]:
ablation_study_fold_metrics = load_ablation_study_metrics()

mae_metric_name = RegressionMetrics.MAE
mse_metric_name = RegressionMetrics.MSE
rmse_metric_name = RegressionMetrics.RMSE

mae_fold_metrics = get_metric_matrix(ablation_study_fold_metrics, mae_metric_name)
mse_fold_metrics = get_metric_matrix(ablation_study_fold_metrics, mse_metric_name)
rmse_fold_metrics = get_metric_matrix(ablation_study_fold_metrics, rmse_metric_name)

# Display loaded MAE fold metrics
print("MAE Fold Metrics:")
display(mae_fold_metrics)

### 3. Statistical Analysis and Hypothesis Testing

The code below performs the formal statistical evaluation of the experiment results to analyze the main effects of both **Algorithm** and **Variant** on the MAE metric. 

The analysis follows a structured pipeline:
1. **Normality Assessment:** The *Shapiro-Wilk* test is used to evaluate the distribution of each metric column.
2. **Global Significance Testing:** Non-parametric *Friedman* tests are applied to detect overall statistically significant differences across models.
3. **Post-hoc Analysis:** The *Conover-Inman* post-hoc test is conducted for pairwise comparisons to isolate specific statistical differences and identify homogeneous model groups.

In [ ]:
# Check for normality of every metrics column
is_everything_normal = check_global_normality(mae_fold_metrics)
print(f"Parametric tests allowed: {is_everything_normal}")


# Aggregate metrics for analysis of algorithm impact (main effect 1)
algo_mae = aggregate_metrics_for_main_effects(mae_fold_metrics, group_by="algorithm")

# Perform Friedman test for algorithm analysis
algo_friedman = perform_friedman_analysis(algo_mae)
print("Friedman Test for Algorithms:\n", algo_friedman)

if algo_friedman['p_unc'].values[0] < ALPHA_SIGNIFICANCE_LEVEL:
    algo_posthoc = perform_conover_posthoc(algo_mae)
    print("Post-hoc Conover-Inman for Algorithms (p-values):")
    display(algo_posthoc)
    algo_posthoc.to_csv("results_posthoc_algorithm.csv")


# Aggregate metrics for analysis of variant impact (main effect 2)
variant_mae = aggregate_metrics_for_main_effects(mae_fold_metrics, group_by="variant")

# Perform Friedman test for variant analysis
variant_friedman = perform_friedman_analysis(variant_mae)
print("\nFriedman Test for Feature Variants:\n", variant_friedman)

if variant_friedman['p_unc'].values[0] < ALPHA_SIGNIFICANCE_LEVEL:
    variant_posthoc = perform_conover_posthoc(variant_mae)
    print("\nPost-hoc Conover-Inman for Variants (p-values):")
    display(variant_posthoc)
    variant_posthoc.to_csv("results_posthoc_variant.csv")